[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sadriica/Curso_ANH/blob/main/modulo2_datos/modulo2_datos.ipynb)

Primera vez en Colab: ver la [guía](https://github.com/Sadriica/Curso_ANH/blob/main/guia_colab.md). Términos: [glosario](https://github.com/Sadriica/Curso_ANH/blob/main/glosario.md).

# Data & GIS para Energía
## Módulo 2: Datos

Curso de 5 horas, nivel básico. Corre completo en Google Colab, sin instalar nada en el equipo.

Contenido:

1. Fundamentos de programación: Python y sus librerías.
2. Transformación de archivos: CSV, Excel, GeoJSON, shapefile, raster.
3. Unificación: llevar fuentes distintas y sistemas de coordenadas distintos a uno solo.
4. Mapeo en malla: pasar los datos a una malla hexagonal (H3) y visualizarlos.
5. Datos raster (GeoTIFF).

Los datos son sintéticos (municipios del norte de Colombia) para que todo corra sin archivos
externos. La lógica es la misma que con datos reales. Ejecute las celdas de arriba a abajo
(Shift+Enter).

En la carpeta `recursos/` (junto a este notebook) están los mismos datos ya generados (CSV, Excel,
GeoJSON, shapefile, la malla H3 y el raster), como respaldo, por si se prefiere cargarlos en vez de
generarlos o si falla algún paso en la sesión.

## 0. Preparación

Colab ya trae pandas, numpy y matplotlib. Falta instalar las de datos geográficos.

In [ ]:
!pip install -q geopandas "h3>=4.1" folium mapclassify rasterio openpyxl

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import h3
import folium
import matplotlib.pyplot as plt

print("pandas", pd.__version__, "| geopandas", gpd.__version__, "| h3", h3.__version__)

## 1. Fundamentos de programación

Python se lee casi como instrucciones en texto. Lo mínimo que hay que conocer:

- **Variable**: un nombre que guarda un valor.
- **Lista**: una colección ordenada de valores.
- **Función**: una acción reutilizable, p. ej. `sum([...])`.
- **Librería**: código ya hecho que se importa (pandas, geopandas).

In [ ]:
velocidad = 7.5          # m/s
potencia_relativa = velocidad ** 3   # la potencia crece con v^3
print(velocidad, "m/s  ->  potencia relativa", potencia_relativa)

In [ ]:
municipios = ["Uribia", "Riohacha", "Maicao"]
vientos = [8.9, 7.2, 6.5]

for nombre, v in zip(municipios, vientos):
    print(nombre, v, "m/s")

### Tablas con pandas

Un DataFrame es una tabla con filas y columnas. Es la forma habitual de trabajar datos:
filtrar, ordenar, calcular columnas.

In [ ]:
tabla = pd.DataFrame({
    "municipio": ["Uribia", "Riohacha", "Maicao"],
    "viento_ms": [8.9, 7.2, 6.5],
    "poblacion": [175000, 280000, 160000],
})
tabla

In [ ]:
tabla["apto_eolico"] = tabla["viento_ms"] >= 7      # columna derivada
tabla.sort_values("viento_ms", ascending=False)

In [ ]:
tabla.plot(x="municipio", y="viento_ms", kind="bar", legend=False)
plt.ylabel("Viento (m/s)")
plt.tight_layout()
plt.show()

## 2. Transformación de archivos

Formatos habituales en GIS:

| Formato | Contenido | Se abre con |
|---|---|---|
| CSV / Excel | tablas | pandas |
| GeoJSON / Shapefile | geometrías (puntos, líneas, polígonos) + atributos | geopandas |
| GeoTIFF | raster: una grilla de valores (p. ej. un mapa de viento) | rasterio |

A continuación se crea un conjunto de datos, se guarda y se convierte entre formatos.

In [ ]:
np.random.seed(42)      # reproducibilidad

municipios = pd.DataFrame({
    "cod_dane": ["44847", "44001", "44430", "44560", "20001", "20011", "47001", "08001"],
    "municipio": ["Uribia", "Riohacha", "Maicao", "Manaure",
                  "Valledupar", "Aguachica", "Santa Marta", "Barranquilla"],
    "lat": [11.71, 11.55, 11.38, 11.78, 10.46, 8.31, 11.24, 10.96],
    "lon": [-71.98, -72.91, -72.24, -72.44, -73.25, -73.63, -74.20, -74.80],
    "viento_ms":  [9.1, 7.3, 6.8, 8.7, 5.2, 4.1, 5.9, 6.2],
    "poblacion":  [175000, 280000, 160000, 100000, 490000, 110000, 515000, 1230000],
    "dist_via_km": [3.2, 0.5, 1.1, 4.8, 0.3, 0.9, 0.4, 0.2],
})
municipios

In [ ]:
municipios.to_csv("municipios.csv", index=False)
pd.read_csv("municipios.csv").head(3)

### Excel

Un Excel se lee y se escribe casi igual que un CSV.

In [ ]:
municipios.to_excel("municipios.xlsx", index=False)
pd.read_excel("municipios.xlsx").head(3)

### De tabla a mapa

Con columnas `lat`/`lon`, la tabla pasa a ser geográfica (un GeoDataFrame). El sistema de
coordenadas EPSG:4326 son grados de latitud/longitud, el más común.

In [ ]:
gdf = gpd.GeoDataFrame(
    municipios,
    geometry=gpd.points_from_xy(municipios["lon"], municipios["lat"]),
    crs="EPSG:4326",
)
gdf.head(3)

In [ ]:
gdf.to_file("municipios.geojson", driver="GeoJSON")
gpd.read_file("municipios.geojson").head(3)

### Shapefile

El shapefile es el formato clásico de los SIG. No es un archivo sino un conjunto de ellos
(`.shp` con la geometría, `.dbf` con los atributos, `.shx` y `.prj` con el índice y el CRS), que
hay que mantener siempre juntos; por eso conviene guardarlo en su propia carpeta.

Tiene una limitación conocida: los nombres de columna se recortan a 10 caracteres.


In [ ]:
import os

os.makedirs("municipios_shp", exist_ok=True)
gdf.to_file("municipios_shp/municipios.shp")

shp = gpd.read_file("municipios_shp/municipios.shp")
print("columnas del original :", list(municipios.columns))
print("columnas del shapefile:", [c for c in shp.columns if c != "geometry"])
shp.head(3)

In [ ]:
ax = gdf.plot(figsize=(6, 6), color="crimson", markersize=40)
for _, r in gdf.iterrows():
    ax.text(r["lon"] + 0.05, r["lat"], r["municipio"], fontsize=8)
plt.show()

## 3. Unificación

Los datos vienen de fuentes distintas y en sistemas de coordenadas distintos. Antes de
analizarlos hay que:

1. Llevar todo al **mismo sistema de coordenadas**.
2. **Unir** las tablas por una llave común (p. ej. el código del municipio).

### Mismo sistema de coordenadas

Colombia suele venir en metros (EPSG:9377, el oficial nacional) o en grados (EPSG:4326).
Si se mezclan sistemas, los puntos no coinciden. La solución es reproyectar.

In [ ]:
gdf_metros = gdf.to_crs("EPSG:9377")             # mismos puntos, ahora en metros
print("en metros:", gdf_metros.geometry.iloc[0])
print("de vuelta a grados:", gdf_metros.to_crs("EPSG:4326").geometry.iloc[0])

### Unir por una llave común

Otra fuente trae otra variable (radiación solar) con el mismo `cod_dane`. Se une con `merge`.

In [ ]:
solar = pd.DataFrame({
    "cod_dane": ["44847", "44001", "44430", "44560", "20001", "20011", "47001", "08001"],
    "radiacion_kwh_m2_dia": [6.1, 5.8, 5.9, 6.0, 5.2, 5.0, 5.4, 5.5],
})

unificado = municipios.merge(solar, on="cod_dane", how="left")
unificado[["municipio", "viento_ms", "poblacion", "radiacion_kwh_m2_dia"]]

## 4. Mapeo en malla (H3)

Comparar municipios de tamaños muy distintos es difícil. Se divide el territorio en celdas
iguales y se ponen los datos en ellas. Aquí se usa H3, una malla de hexágonos creada por Uber:
las celdas tienen tamaño casi igual y cada una tiene un código único, lo que facilita comparar
y cruzar fuentes. La resolución define el tamaño del hexágono
(mayor resolución, celdas más pequeñas).

In [ ]:
RES = 5      # hexagonos de ~8 km de lado
unificado["h3_index"] = [h3.latlng_to_cell(lat, lon, RES)
                         for lat, lon in zip(municipios["lat"], municipios["lon"])]
unificado[["municipio", "lat", "lon", "h3_index"]]

### Agregar por celda

Si varios puntos caen en la misma celda, se resumen (aquí, promedio del viento). Con miles de
datos el paso es idéntico.

In [ ]:
por_celda = unificado.groupby("h3_index", as_index=False).agg(
    viento_ms=("viento_ms", "mean"),
    dist_via_km=("dist_via_km", "mean"),
    radiacion=("radiacion_kwh_m2_dia", "mean"),
    municipios=("municipio", lambda x: ", ".join(x)),
)
# esta malla (celdas H3 + variables) es lo que pasa al Modulo 3
por_celda.to_parquet("malla_h3.parquet", index=False)
por_celda

### Visualizar la malla

Cada celda H3 se convierte en su polígono hexagonal y se pinta según el viento
(azul = poco, rojo = mucho). Este es el tipo de mapa que produce el análisis.

In [ ]:
import branca.colormap as cm

vmin, vmax = por_celda["viento_ms"].min(), por_celda["viento_ms"].max()
colormap = cm.LinearColormap(["blue", "yellow", "red"], vmin=vmin, vmax=vmax)
colormap.caption = "Velocidad del viento (m/s)"

m = folium.Map(location=[11.0, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, row in por_celda.iterrows():
    borde = h3.cell_to_boundary(row["h3_index"])        # lista de (lat, lon)
    folium.Polygon(
        locations=[[lat, lon] for lat, lon in borde],
        color="grey", weight=1,
        fill=True, fill_color=colormap(row["viento_ms"]), fill_opacity=0.7,
        tooltip=f"{row['municipios']}: {row['viento_ms']:.1f} m/s",
    ).add_to(m)
colormap.add_to(m)
m

### Rellenar un área completa

También se pueden generar todos los hexágonos que cubren una zona, no solo donde hay puntos,
para tener una malla continua sobre la región de estudio.

In [ ]:
zona = {
    "type": "Polygon",
    "coordinates": [[
        [-75.2, 8.0], [-71.5, 8.0], [-71.5, 12.2], [-75.2, 12.2], [-75.2, 8.0]
    ]],
}
# el poligono va en orden GeoJSON [lon, lat]
celdas = h3.geo_to_cells(zona, RES)
print("celdas en la zona:", len(celdas))

m2 = folium.Map(location=[10.0, -73.3], zoom_start=6, tiles="CartoDB positron")
for c in celdas:
    borde = h3.cell_to_boundary(c)
    folium.Polygon(locations=[[lat, lon] for lat, lon in borde],
                   color="steelblue", weight=1, fill=True, fill_opacity=0.1).add_to(m2)
m2

## 5. Datos raster (GeoTIFF)

Un raster es una grilla de valores, como una foto donde cada pixel tiene un número (por
ejemplo, la velocidad del viento en cada punto del territorio). Es el otro gran tipo de dato
geográfico, además de las tablas y los vectores.

A continuación se crea un raster pequeño de viento, se guarda como GeoTIFF, se lee y se consulta
el valor en la ubicación de cada municipio.

In [ ]:
import rasterio
from rasterio.transform import from_bounds

W, H = 60, 60                                   # 60x60 pixeles
lon0, lon1, lat0, lat1 = -75.5, -71.0, 8.0, 12.6   # limites de la zona
col = np.linspace(0, 1, W)[None, :]             # 0 al oeste -> 1 al este
row = np.linspace(0, 1, H)[:, None]             # 0 al norte (arriba) -> 1 al sur
viento_grid = (3 + 6 * (0.5 * col + 0.5 * (1 - row))).astype("float32")   # mas viento al NE

transform = from_bounds(lon0, lat0, lon1, lat1, W, H)
with rasterio.open("viento.tif", "w", driver="GTiff", height=H, width=W, count=1,
                   dtype="float32", crs="EPSG:4326", transform=transform) as dst:
    dst.write(viento_grid, 1)
print("raster viento.tif guardado (", W, "x", H, "pixeles )")

In [ ]:
# Leer el raster y verlo
with rasterio.open("viento.tif") as src:
    import rasterio.plot
    rasterio.plot.show(src, cmap="viridis", title="Viento (raster)")

### Muestrear el raster en los municipios

Consultar el valor del raster en cada punto (lon, lat) se llama *sampling*. Así se cruza un
raster con puntos.

In [ ]:
puntos = [(lon, lat) for lon, lat in zip(municipios["lon"], municipios["lat"])]
with rasterio.open("viento.tif") as src:
    muestras = [float(v[0]) for v in src.sample(puntos)]

municipios["viento_raster"] = muestras
municipios[["municipio", "viento_ms", "viento_raster"]]

## Actividad individual

Usando la tabla `unificado`, cree una columna `apto` que sea `True` donde el municipio tenga
buen viento y buen acceso: `viento_ms >= 7` y `dist_via_km <= 2`. Muestre solo los aptos.

Escriba su código en la celda siguiente. Más abajo está la solución.

In [ ]:
# Escriba su codigo aqui


<details><summary>Ver solución</summary>

```python
unificado["apto"] = (unificado["viento_ms"] >= 7) & (unificado["dist_via_km"] <= 2)
unificado.loc[unificado["apto"], ["municipio", "viento_ms", "dist_via_km"]]
# Riohacha cumple (viento 7.3, via 0.5). Uribia y Manaure tienen mas viento pero via > 2.
```
</details>

Ejercicio adicional: cambie `RES = 5` por `RES = 6` en la sección 4 y vuelva a ejecutar. Observe
el efecto en el tamaño de los hexágonos y en el número de celdas de la zona.

## Cierre

En este módulo:

1. Manejamos datos con Python y pandas.
2. Leímos y convertimos archivos (CSV, Excel, GeoJSON, shapefile).
3. Unificamos fuentes: mismo sistema de coordenadas (reproyectar) y unión por llave (`merge`).
4. Mapeamos todo en una malla H3 y lo visualizamos.
5. Leímos un raster (GeoTIFF) y lo muestreamos en los municipios.

Con todo en la misma malla y comparable, el Módulo 3 combina varias variables (viento,
distancia a vías, población) para decidir dónde conviene un proyecto de energía: es el análisis
multicriterio, con AHP.